# Data collection
Collect and validate the external datasets used in the project before combining them into a modelling dataset.

## 1. Setup

In [5]:
from pathlib import Path
import json

import pandas as pd
import requests

In [6]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carbon intensity 

### 2.1 1-day prototype
Test the API and understand its structure using a single day before scaling the collection to the full historical period.

In [7]:
TEST_DATE = "2025-01-01"

url = f"https://api.carbonintensity.org.uk/intensity/date/{TEST_DATE}"

response = requests.get(url, timeout=30)
response.raise_for_status()

carbon_raw = response.json()
raw_path = RAW_DIR / f"carbon_intensity_{TEST_DATE}.json"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(carbon_raw, f, indent=2)

In [8]:
carbon_df = pd.json_normalize(carbon_raw["data"])

carbon_df.head()

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2025-01-01T00:00Z,2025-01-01T00:30Z,49,55,low
1,2025-01-01T00:30Z,2025-01-01T01:00Z,52,54,low
2,2025-01-01T01:00Z,2025-01-01T01:30Z,56,53,low
3,2025-01-01T01:30Z,2025-01-01T02:00Z,53,53,low
4,2025-01-01T02:00Z,2025-01-01T02:30Z,53,47,low


In [9]:
carbon_df = carbon_df.rename(
    columns={
        "intensity.forecast": "carbon_forecast_gco2_kwh",
        "intensity.actual": "carbon_actual_gco2_kwh",
        "intensity.index": "carbon_index",
    }
)

In [10]:
carbon_df["from"] = pd.to_datetime(carbon_df["from"], utc=True)
carbon_df["to"] = pd.to_datetime(carbon_df["to"], utc=True)

### 2.2 Validate prototype


In [11]:
carbon_df.head()

,from,to,carbon_forecast_gco2_kwh,carbon_actual_gco2_kwh,carbon_index
0,2025-01-01 00:00:00+00:00,2025-01-01 00:30:00+00:00,49,55,low
1,2025-01-01 00:30:00+00:00,2025-01-01 01:00:00+00:00,52,54,low
2,2025-01-01 01:00:00+00:00,2025-01-01 01:30:00+00:00,56,53,low
3,2025-01-01 01:30:00+00:00,2025-01-01 02:00:00+00:00,53,53,low
4,2025-01-01 02:00:00+00:00,2025-01-01 02:30:00+00:00,53,47,low


In [12]:
carbon_df.tail()

,from,to,carbon_forecast_gco2_kwh,carbon_actual_gco2_kwh,carbon_index
43,2025-01-01 21:30:00+00:00,2025-01-01 22:00:00+00:00,81,77,low
44,2025-01-01 22:00:00+00:00,2025-01-01 22:30:00+00:00,82,76,low
45,2025-01-01 22:30:00+00:00,2025-01-01 23:00:00+00:00,94,65,low
46,2025-01-01 23:00:00+00:00,2025-01-01 23:30:00+00:00,83,63,low
47,2025-01-01 23:30:00+00:00,2025-01-02 00:00:00+00:00,77,67,low


In [13]:
carbon_df.info()

print("Shape:", carbon_df.shape)

print("\nMissing values:")
print(carbon_df.isna().sum())

print("\nDuplicate timestamps:")
print(carbon_df["from"].duplicated().sum())

print("\nInterval lengths:")
print((carbon_df["to"] - carbon_df["from"]).value_counts())

print("\nTime range:")
print(carbon_df["from"].min(), "to", carbon_df["to"].max())

print("\nCarbon actual summary:")
print(carbon_df["carbon_actual_gco2_kwh"].describe())

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   from                      48 non-null     datetime64[us, UTC]
 1   to                        48 non-null     datetime64[us, UTC]
 2   carbon_forecast_gco2_kwh  48 non-null     int64              
 3   carbon_actual_gco2_kwh    48 non-null     int64              
 4   carbon_index              48 non-null     str                
dtypes: datetime64[us, UTC](2), int64(2), str(1)
memory usage: 2.0 KB
Shape: (48, 5)

Missing values:
from                        0
to                          0
carbon_forecast_gco2_kwh    0
carbon_actual_gco2_kwh      0
carbon_index                0
dtype: int64

Duplicate timestamps:
0

Interval lengths:
0 days 00:30:00    48
Name: count, dtype: int64

Time range:
2025-01-01 00:00:00+00:00 to 2025-01-02 00:00:00+00:00

Carbon actual 

In [14]:
carbon_df[
    [
        "carbon_forecast_gco2_kwh",
        "carbon_actual_gco2_kwh",
    ]
].describe()

,carbon_forecast_gco2_kwh,carbon_actual_gco2_kwh
count,48.000000,48.000000
mean,58.875000,59.291667
std,22.732136,21.149879
min,23.000000,35.000000
25%,42.000000,42.750000
50%,48.500000,50.000000
75%,82.250000,78.750000
max,98.000000,95.000000


### 2.3 Historical collection

In [15]:
def carbon_day(start, end):
    start = pd.to_datetime(start, utc=True)
    end = pd.to_datetime(end, utc=True)

    start_str = start.strftime("%Y-%m-%dT%H:%MZ")
    end_str = end.strftime("%Y-%m-%dT%H:%MZ")

    url = f"https://api.carbonintensity.org.uk/intensity/{start_str}/{end_str}"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    raw_data = response.json()

    df = pd.json_normalize(raw_data["data"])

    df = df.rename(
        columns={
            "intensity.forecast": "carbon_forecast_gco2_kwh",
            "intensity.actual": "carbon_actual_gco2_kwh",
            "intensity.index": "carbon_index",
        }
    )

    df = df[
        [
            "from",
            "to",
            "carbon_forecast_gco2_kwh",
            "carbon_actual_gco2_kwh",
            "carbon_index",
        ]
    ]

    df["from"] = pd.to_datetime(df["from"], utc=True)
    df["to"] = pd.to_datetime(df["to"], utc=True)

    # API may return the preceding half-hour interval
    df = df[
        (df["from"] >= start) &
        (df["from"] < end)
    ].copy()

    return df


In [16]:
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

carbon_frames = []

for date in dates:
    start = date
    end = date + pd.Timedelta(days=1)

    daily_df = carbon_day(start, end)
    carbon_frames.append(daily_df)

carbon_history = pd.concat(
    carbon_frames,
    ignore_index=True
)

KeyboardInterrupt: 

### 2.4 Clean, validate and save

The historical carbon-intensity dataset contains 35,057 observations compared with 35,088 expected half-hour intervals.

Two data-quality issues were identified:

1. 31 missing timestamps: a continuous 15.5-hour block is absent from 2024-06-11 23:00 UTC to 2024-06-12 14:00 UTC. 

2. 43 missing forecast values: these observations contain valid timestamps and actual carbon intensity values, but `carbon_forecast_gco2_kwh` is missing.

No duplicate timestamps were found. All available observations are chronologically ordered and represent 30-minute intervals. Actual carbon-intensity values contain no missing values.

In [ ]:
expected_times = pd.date_range(
    start="2024-01-01",
    end="2026-01-01",
    freq="30min",
    inclusive="left",
    tz="UTC"
)

missing_times = expected_times.difference(carbon_history["from"])

print("Expected rows:", len(expected_times))
print("Actual rows:", len(carbon_history))
print("Start:", carbon_history["from"].min())
print("End:", carbon_history["to"].max())
print("Missing timestamps:", len(missing_times))
print("Duplicate timestamps:", carbon_history["from"].duplicated().sum())

print(carbon_history.isna().sum())

print(
    "Chronological:",
    carbon_history["from"].is_monotonic_increasing
)

print(
    (carbon_history["to"] - carbon_history["from"]).value_counts()
)

print(
    carbon_history["from"].diff().value_counts().head()
)

print(
    carbon_history["carbon_actual_gco2_kwh"].describe()
)

print(
    carbon_history["carbon_forecast_gco2_kwh"].describe()
)

print(
    carbon_history["carbon_index"].value_counts(dropna=False)
)

Expected rows: 35088
Actual rows: 35057
Start: 2024-01-01 00:00:00+00:00
End: 2026-01-01 00:00:00+00:00
Missing timestamps: 31
Duplicate timestamps: 0
from                         0
to                           0
carbon_forecast_gco2_kwh    43
carbon_actual_gco2_kwh       0
carbon_index                 0
dtype: int64
Chronological: True
0 days 00:30:00    35057
Name: count, dtype: int64
from
0 days 00:30:00    35055
0 days 16:00:00        1
Name: count, dtype: int64
count    35057.000000
mean       127.142568
std         58.793737
min         14.000000
25%         76.000000
50%        120.000000
75%        170.000000
max        303.000000
Name: carbon_actual_gco2_kwh, dtype: float64
count    35014.000000
mean       127.058462
std         60.478509
min         13.000000
25%         75.000000
50%        120.000000
75%        172.000000
max        309.000000
Name: carbon_forecast_gco2_kwh, dtype: float64
carbon_index
low          14441
moderate     13652
high          6231
very high      

In [ ]:
missing_times = expected_times.difference(carbon_history["from"])

missing_times

DatetimeIndex(['2024-06-11 23:00:00+00:00', '2024-06-11 23:30:00+00:00',
               '2024-06-12 00:00:00+00:00', '2024-06-12 00:30:00+00:00',
               '2024-06-12 01:00:00+00:00', '2024-06-12 01:30:00+00:00',
               '2024-06-12 02:00:00+00:00', '2024-06-12 02:30:00+00:00',
               '2024-06-12 03:00:00+00:00', '2024-06-12 03:30:00+00:00',
               '2024-06-12 04:00:00+00:00', '2024-06-12 04:30:00+00:00',
               '2024-06-12 05:00:00+00:00', '2024-06-12 05:30:00+00:00',
               '2024-06-12 06:00:00+00:00', '2024-06-12 06:30:00+00:00',
               '2024-06-12 07:00:00+00:00', '2024-06-12 07:30:00+00:00',
               '2024-06-12 08:00:00+00:00', '2024-06-12 08:30:00+00:00',
               '2024-06-12 09:00:00+00:00', '2024-06-12 09:30:00+00:00',
               '2024-06-12 10:00:00+00:00', '2024-06-12 10:30:00+00:00',
               '2024-06-12 11:00:00+00:00', '2024-06-12 11:30:00+00:00',
               '2024-06-12 12:00:00+00:00', '2024-0

In [ ]:
forecast_missing = carbon_history[
    carbon_history["carbon_forecast_gco2_kwh"].isna()
]

forecast_missing

,from,to,carbon_forecast_gco2_kwh,carbon_actual_gco2_kwh,carbon_index
18111,2025-01-12 23:00:00+00:00,2025-01-12 23:30:00+00:00,NaN,126,moderate
18112,2025-01-12 23:30:00+00:00,2025-01-13 00:00:00+00:00,NaN,126,moderate
18113,2025-01-13 00:00:00+00:00,2025-01-13 00:30:00+00:00,NaN,129,moderate
18114,2025-01-13 00:30:00+00:00,2025-01-13 01:00:00+00:00,NaN,126,moderate
18115,2025-01-13 01:00:00+00:00,2025-01-13 01:30:00+00:00,NaN,123,moderate
18116,2025-01-13 01:30:00+00:00,2025-01-13 02:00:00+00:00,NaN,125,moderate
18117,2025-01-13 02:00:00+00:00,2025-01-13 02:30:00+00:00,NaN,124,moderate
18118,2025-01-13 02:30:00+00:00,2025-01-13 03:00:00+00:00,NaN,125,moderate
18119,2025-01-13 03:00:00+00:00,2025-01-13 03:30:00+00:00,NaN,125,moderate
18120,2025-01-13 03:30:00+00:00,2025-01-13 04:00:00+00:00,NaN,128,moderate


In [ ]:
full_timeline = pd.DataFrame({
    "from": expected_times
})

carbon_clean = full_timeline.merge(
    carbon_history,
    on="from",
    how="left"
)

carbon_clean["to"] = (
    carbon_clean["from"] + pd.Timedelta(minutes=30)
)

In [ ]:
print("Shape:", carbon_clean.shape)

print("Missing values:")
print(carbon_clean.isna().sum())

print(
    "Duplicate timestamps:",
    carbon_clean["from"].duplicated().sum()
)

print(
    "Chronological:",
    carbon_clean["from"].is_monotonic_increasing
)

Shape: (35088, 5)
Missing values:
from                         0
to                           0
carbon_forecast_gco2_kwh    74
carbon_actual_gco2_kwh      31
carbon_index                31
dtype: int64
Duplicate timestamps: 0
Chronological: True


In [ ]:
processed_path = (
    PROCESSED_DIR / "carbon_intensity_2024_2025.csv"
)

carbon_clean.to_csv(
    processed_path,
    index=False
)


Saved: c:\Users\Serface\data-centre-energy-optimisation\data\processed\carbon_intensity_2024_2025.csv


## 3. Electricity price


### 3.1 1-day prototype

In [19]:
TEST_DATE = "2025-01-01"

start = pd.Timestamp(TEST_DATE, tz="UTC")
end = start + pd.Timedelta(days=1)

url = "https://data.elexon.co.uk/bmrs/api/v1/balancing/pricing/market-index"

params = {
    "from": start.strftime("%Y-%m-%dT%H:%MZ"),
    "to": end.strftime("%Y-%m-%dT%H:%MZ")
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

price_raw = response.json()

price_raw['data'][0]
    

{'startTime': '2025-01-02T00:00:00Z',
 'dataProvider': 'APXMIDP',
 'settlementDate': '2025-01-02',
 'settlementPeriod': 1,
 'price': 110.4,
 'volume': 1624.1}

In [20]:
price_df = pd.DataFrame(price_raw["data"])

price_df.head()

,startTime,dataProvider,settlementDate,settlementPeriod,price,volume
0,2025-01-02T00:00:00Z,APXMIDP,2025-01-02,1,110.40,1624.1
1,2025-01-02T00:00:00Z,N2EXMIDP,2025-01-02,1,0.00,0.0
2,2025-01-01T23:30:00Z,APXMIDP,2025-01-01,48,106.05,1324.0
3,2025-01-01T23:30:00Z,N2EXMIDP,2025-01-01,48,0.00,0.0
4,2025-01-01T23:00:00Z,APXMIDP,2025-01-01,47,107.77,1265.2


In [22]:
price_df["startTime"] = pd.to_datetime(price_df["startTime"], utc=True)

price_df = (
    price_df[
        (price_df["dataProvider"] == "APXMIDP") &
        (price_df["startTime"] >= start) &
        (price_df["startTime"] < end)
    ]
    [["startTime", "price", "volume"]]
    .sort_values("startTime")
    .reset_index(drop=True)
)

print("Shape:", price_df.shape)
display(price_df.head())


Shape: (48, 3)


,startTime,price,volume
0,2025-01-01 00:00:00+00:00,9.07,1470.95
1,2025-01-01 00:30:00+00:00,23.51,1792.80
2,2025-01-01 01:00:00+00:00,9.11,1900.55
3,2025-01-01 01:30:00+00:00,8.94,1891.00
4,2025-01-01 02:00:00+00:00,29.65,2141.60


### 3.2 Validate prototype

In [23]:
print("Shape:", price_df.shape)

print("\nMissing values:")
print(price_df.isna().sum())

print("\nDuplicate timestamps:")
print(price_df["startTime"].duplicated().sum())

print("\nTime range:")
print(price_df["startTime"].min(), "to", price_df["startTime"].max())

print("\nPrice summary:")
print(price_df["price"].describe())

print("\nVolume summary:")
print(price_df["volume"].describe())


Shape: (48, 3)

Missing values:
startTime    0
price        0
volume       0
dtype: int64

Duplicate timestamps:
0

Time range:
2025-01-01 00:00:00+00:00 to 2025-01-01 23:30:00+00:00

Price summary:
count     48.000000
mean      55.903542
std       52.188545
min      -14.330000
25%        5.807500
50%       37.610000
75%      108.522500
max      126.730000
Name: price, dtype: float64

Volume summary:
count      48.000000
mean     1929.066667
std       365.290564
min      1265.200000
25%      1700.712500
50%      1897.750000
75%      2173.175000
max      2752.700000
Name: volume, dtype: float64


### 3.3 Historical collection

In [24]:
def price_day(start, end):
    start = pd.to_datetime(start, utc=True)
    end = pd.to_datetime(end, utc=True)

    params = {
        "from": start.strftime("%Y-%m-%dT%H:%MZ"),
        "to": end.strftime("%Y-%m-%dT%H:%MZ")
    }

    response = requests.get(
        "https://data.elexon.co.uk/bmrs/api/v1/balancing/pricing/market-index",
        params=params,
        timeout=30
    )
    response.raise_for_status()

    df = pd.DataFrame(response.json()["data"])

    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)

    df = df[
        (df["dataProvider"] == "APXMIDP") &
        (df["startTime"] >= start) &
        (df["startTime"] < end)
    ]

    return df[
        ["startTime", "price", "volume"]
    ].sort_values("startTime").reset_index(drop=True)

In [26]:
price_frames = []

for date in dates:
    start = date
    end = date + pd.Timedelta(days=1)

    daily_df = price_day(start, end)
    price_frames.append(daily_df)

price_history = pd.concat(
    price_frames,
    ignore_index=True
)

print("Collected observations:", len(price_history))


Collected observations: 35087


### 3.4 Clean, validate and save

In [65]:
expected_times = pd.date_range(
    start="2024-01-01",
    end="2026-01-01",
    freq="30min",
    inclusive="left",
    tz="UTC"
)

missing_price_times = expected_times.difference(price_history["startTime"])

print("Expected rows:", len(expected_times))
print("Actual rows:", len(price_history))
print("Missing timestamps:", len(missing_price_times))
print("Duplicate timestamps:", price_history["startTime"].duplicated().sum())

print("\nMissing values:")
print(price_history.isna().sum())

print("\nChronological:", price_history["startTime"].is_monotonic_increasing)

print("\nPrice summary:")
print(price_history["price"].describe())

print("\nVolume summary:")
print(price_history["volume"].describe())

# %%
missing_price_times

# %%
assert price_history["startTime"].is_unique, "Duplicate timestamps found"

full_timeline = pd.DataFrame({
    "startTime": expected_times
})

price_clean = full_timeline.merge(
    price_history,
    on="startTime",
    how="left",
    validate="one_to_one"
)

# %%
print("Shape:", price_clean.shape)

print("\nMissing values:")
print(price_clean.isna().sum())

print("\nDuplicate timestamps:",
      price_clean["startTime"].duplicated().sum())

print("\nChronological:",
      price_clean["startTime"].is_monotonic_increasing)

# %%
processed_path = (
    PROCESSED_DIR / "electricity_price_2024_2025.csv"
)

price_clean.to_csv(processed_path, index=False)


Expected rows: 35088
Actual rows: 35087
Missing timestamps: 1
Duplicate timestamps: 0

Missing values:
startTime    0
price        0
volume       0
dtype: int64

Chronological: True

Price summary:
count    35087.000000
mean        75.603490
std         36.281145
min       -102.920000
25%         63.525000
50%         78.280000
75%         92.500000
max       1352.900000
Name: price, dtype: float64

Volume summary:
count    35087.000000
mean      2317.684738
std        812.122716
min          0.000000
25%       1737.875000
50%       2228.350000
75%       2814.650000
max       8004.700000
Name: volume, dtype: float64
Shape: (35088, 3)

Missing values:
startTime    0
price        1
volume       1
dtype: int64

Duplicate timestamps: 0

Chronological: True


In [66]:
t = pd.Timestamp("2024-04-13 07:00", tz="UTC")

print(price_clean.loc[price_clean["startTime"].eq(t)])
print(price_clean[["price", "volume"]].isna().sum())

                     startTime  price  volume
4958 2024-04-13 07:00:00+00:00    NaN     NaN
price     1
volume    1
dtype: int64


In [67]:
price_clean.to_csv(
    PROCESSED_DIR / "electricity_price_2024_2025.csv",
    index=False
)

## 4. Electricity demand

### 4.1 1-day prototype

In [34]:
TEST_DATE = "2025-01-01"

url = "https://data.elexon.co.uk/bmrs/api/v1/demand/outturn"

params = {
    "settlementDateFrom": TEST_DATE,
    "settlementDateTo": TEST_DATE
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

demand_raw = response.json()

demand_raw["data"][0]


{'publishTime': '2025-01-01T00:30:00Z',
 'startTime': '2025-01-01T00:00:00Z',
 'settlementDate': '2025-01-01',
 'settlementPeriod': 1,
 'initialDemandOutturn': 20949,
 'initialTransmissionSystemDemandOutturn': 26483}

In [35]:
demand_df = pd.DataFrame(demand_raw["data"])

display(demand_df.head())
print(demand_df.columns.tolist())
print("Shape:", demand_df.shape)


,publishTime,startTime,settlementDate,settlementPeriod,initialDemandOutturn,initialTransmissionSystemDemandOutturn
0,2025-01-01T00:30:00Z,2025-01-01T00:00:00Z,2025-01-01,1,20949,26483
1,2025-01-01T01:00:00Z,2025-01-01T00:30:00Z,2025-01-01,2,21129,26474
2,2025-01-01T01:30:00Z,2025-01-01T01:00:00Z,2025-01-01,3,21266,26123
3,2025-01-01T02:00:00Z,2025-01-01T01:30:00Z,2025-01-01,4,20356,25410
4,2025-01-01T02:30:00Z,2025-01-01T02:00:00Z,2025-01-01,5,19508,25323


['publishTime', 'startTime', 'settlementDate', 'settlementPeriod', 'initialDemandOutturn', 'initialTransmissionSystemDemandOutturn']
Shape: (48, 6)


### 4.2 Validate prototype


In [48]:
demand_df = pd.DataFrame(demand_raw["data"])

demand_df["startTime"] = pd.to_datetime(
    demand_df["startTime"], utc=True
)

demand_df = demand_df[
    ["startTime", "initialDemandOutturn"]
].sort_values("startTime").reset_index(drop=True)

demand_df.head()



,startTime,initialDemandOutturn
0,2025-01-01 00:00:00+00:00,20949
1,2025-01-01 00:30:00+00:00,21129
2,2025-01-01 01:00:00+00:00,21266
3,2025-01-01 01:30:00+00:00,20356
4,2025-01-01 02:00:00+00:00,19508


In [50]:
print("Shape:", demand_df.shape)
print("\nMissing values:")
print(demand_df.isna().sum())
print("\nDuplicate timestamps:", demand_df["startTime"].duplicated().sum())
print("\nTime range:", demand_df["startTime"].min(), "to", demand_df["startTime"].max())
print("\nDemand summary:")
print(demand_df["initialDemandOutturn"].describe())



Shape: (48, 2)

Missing values:
startTime               0
initialDemandOutturn    0
dtype: int64

Duplicate timestamps: 0

Time range: 2025-01-01 00:00:00+00:00 to 2025-01-01 23:30:00+00:00

Demand summary:
count       48.000000
mean     25334.750000
std       5538.128632
min      17283.000000
25%      19855.250000
50%      25335.000000
75%      30040.250000
max      33919.000000
Name: initialDemandOutturn, dtype: float64


### 4.3 Historical collection

In [52]:
def demand_day(date):
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")

    response = requests.get(
        "https://data.elexon.co.uk/bmrs/api/v1/demand/outturn",
        params={
            "settlementDateFrom": date_str,
            "settlementDateTo": date_str
        },
        timeout=30
    )
    response.raise_for_status()

    df = pd.DataFrame(response.json()["data"])

    df["startTime"] = pd.to_datetime(
        df["startTime"], utc=True
    )

    return df[
        ["startTime", "initialDemandOutturn"]
    ].sort_values("startTime").reset_index(drop=True)


In [53]:
demand_frames = []
failed_dates = []

for date in dates:
    try:
        daily_df = demand_day(date)
        demand_frames.append(daily_df)
    except requests.exceptions.RequestException:
        failed_dates.append(date)

demand_history = pd.concat(
    demand_frames,
    ignore_index=True
).sort_values("startTime").reset_index(drop=True)

print("Collected observations:", len(demand_history))
print("Failed dates:", len(failed_dates))


Collected observations: 35088
Failed dates: 0


### 4.4 Clean validate and save

In [54]:
missing_demand_times = expected_times.difference(demand_history["startTime"])

print("Expected rows:", len(expected_times))
print("Actual rows:", len(demand_history))
print("Missing timestamps:", len(missing_demand_times))
print("Duplicate timestamps:", demand_history["startTime"].duplicated().sum())

print("\nMissing values:")
print(demand_history.isna().sum())

print("\nChronological:", demand_history["startTime"].is_monotonic_increasing)

print("\nDemand summary:")
print(demand_history["initialDemandOutturn"].describe())

# %%
missing_demand_times

# %%
full_timeline = pd.DataFrame({
    "startTime": expected_times
})

demand_clean = full_timeline.merge(
    demand_history,
    on="startTime",
    how="left",
    validate="one_to_one"
)

# %%
print("Shape:", demand_clean.shape)

print("\nMissing values:")
print(demand_clean.isna().sum())

print("\nDuplicate timestamps:",
      demand_clean["startTime"].duplicated().sum())

print("\nChronological:",
      demand_clean["startTime"].is_monotonic_increasing)

# %%
processed_path = (
    PROCESSED_DIR / "electricity_demand_2024_2025.csv"
)

demand_clean.to_csv(
    processed_path,
    index=False
)


Expected rows: 35088
Actual rows: 35088
Missing timestamps: 0
Duplicate timestamps: 0

Missing values:
startTime               0
initialDemandOutturn    0
dtype: int64

Chronological: True

Demand summary:
count    35088.000000
mean     26186.232957
std       6042.226824
min      12812.000000
25%      21477.750000
50%      25126.000000
75%      30005.500000
max      45851.000000
Name: initialDemandOutturn, dtype: float64
Shape: (35088, 2)

Missing values:
startTime               0
initialDemandOutturn    0
dtype: int64

Duplicate timestamps: 0

Chronological: True


## 5. Electricity generation

### 5.1 1-day prototype

In [55]:
TEST_DATE = "2025-01-01"

url = "https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH"

params = {
    "settlementDateFrom": TEST_DATE,
    "settlementDateTo": TEST_DATE
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

generation_raw = response.json()

generation_raw["data"][0]


{'dataset': 'FUELHH',
 'publishTime': '2025-01-02T00:00:00Z',
 'startTime': '2025-01-01T23:30:00Z',
 'settlementDate': '2025-01-01',
 'settlementPeriod': 48,
 'fuelType': 'BIOMASS',
 'generation': 2459}

In [56]:
generation_df = pd.DataFrame(generation_raw["data"])

print("Shape:", generation_df.shape)
print("\nColumns:", generation_df.columns.tolist())
print("\nFuel types:", generation_df["fuelType"].unique())

display(generation_df.head())


Shape: (960, 7)

Columns: ['dataset', 'publishTime', 'startTime', 'settlementDate', 'settlementPeriod', 'fuelType', 'generation']

Fuel types: <StringArray>
['BIOMASS',    'CCGT',    'COAL', 'INTELEC',   'INTEW',   'INTFR', 'INTGRNL',
 'INTIFA2',  'INTIRL',  'INTNED',  'INTNEM',  'INTNSL',  'INTVKL',  'NPSHYD',
 'NUCLEAR',    'OCGT',     'OIL',   'OTHER',      'PS',    'WIND']
Length: 20, dtype: str


,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation
0,FUELHH,2025-01-02T00:00:00Z,2025-01-01T23:30:00Z,2025-01-01,48,BIOMASS,2459
1,FUELHH,2025-01-02T00:00:00Z,2025-01-01T23:30:00Z,2025-01-01,48,CCGT,2739
2,FUELHH,2025-01-02T00:00:00Z,2025-01-01T23:30:00Z,2025-01-01,48,COAL,0
3,FUELHH,2025-01-02T00:00:00Z,2025-01-01T23:30:00Z,2025-01-01,48,INTELEC,0
4,FUELHH,2025-01-02T00:00:00Z,2025-01-01T23:30:00Z,2025-01-01,48,INTEW,-474


### 5.2 Validate prototype

In [57]:
generation_df["startTime"] = pd.to_datetime(
    generation_df["startTime"], utc=True
)

generation_wide = generation_df.pivot(
    index="startTime",
    columns="fuelType",
    values="generation"
).reset_index()

generation_wide.columns.name = None

generation_wide = generation_wide.sort_values(
    "startTime"
).reset_index(drop=True)

print("Shape:", generation_wide.shape)
display(generation_wide.head())


Shape: (48, 21)


,startTime,BIOMASS,CCGT,COAL,INTELEC,INTEW,INTFR,INTGRNL,INTIFA2,INTIRL,...,INTNEM,INTNSL,INTVKL,NPSHYD,NUCLEAR,OCGT,OIL,OTHER,PS,WIND
0,2025-01-01 00:00:00+00:00,962,3710,0,0,-382,-1516,0,-1026,-228,...,-564,88,246,744,5062,1,0,223,-1218,15799
1,2025-01-01 00:30:00+00:00,1097,3934,0,0,-282,-1512,0,-1022,-122,...,-562,52,264,745,5059,1,0,281,-1246,15317
2,2025-01-01 01:00:00+00:00,1106,3769,0,0,-386,-1426,0,-920,-120,...,-198,62,418,744,5056,0,0,293,-1208,14992
3,2025-01-01 01:30:00+00:00,1083,3719,0,0,-334,-1426,0,-916,-120,...,-200,32,422,745,5057,0,0,223,-1448,14580
4,2025-01-01 02:00:00+00:00,1007,3674,0,0,-364,-1542,0,-898,-120,...,-630,-288,286,737,5057,1,0,229,-1374,14685


In [58]:
print("Shape:", generation_wide.shape)

print("\nMissing values:")
print(generation_wide.isna().sum())

print("\nDuplicate timestamps:")
print(generation_wide["startTime"].duplicated().sum())

print("\nTime range:")
print(generation_wide["startTime"].min(), "to", generation_wide["startTime"].max())

print("\nGeneration summary:")
print(generation_wide.drop(columns="startTime").describe())


Shape: (48, 21)

Missing values:
startTime    0
BIOMASS      0
CCGT         0
COAL         0
INTELEC      0
INTEW        0
INTFR        0
INTGRNL      0
INTIFA2      0
INTIRL       0
INTNED       0
INTNEM       0
INTNSL       0
INTVKL       0
NPSHYD       0
NUCLEAR      0
OCGT         0
OIL          0
OTHER        0
PS           0
WIND         0
dtype: int64

Duplicate timestamps:
0

Time range:
2025-01-01 00:00:00+00:00 to 2025-01-01 23:30:00+00:00

Generation summary:
           BIOMASS         CCGT  COAL  INTELEC       INTEW       INTFR  \
count    48.000000    48.000000  48.0     48.0   48.000000    48.00000   
mean   1530.562500  3708.041667   0.0      0.0 -413.625000   253.37500   
std     738.330806  1552.633589   0.0      0.0  128.602079  1420.99271   
min     765.000000  1826.000000   0.0      0.0 -532.000000 -1546.00000   
25%     827.750000  2636.500000   0.0      0.0 -528.000000 -1468.00000   
50%    1142.000000  3017.000000   0.0      0.0 -466.000000  1486.00000   
75%    

### 5.3 Historical collection

In [59]:
def generation_day(date):
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")

    response = requests.get(
        "https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH",
        params={
            "settlementDateFrom": date_str,
            "settlementDateTo": date_str
        },
        timeout=30
    )
    response.raise_for_status()

    df = pd.DataFrame(response.json()["data"])

    df["startTime"] = pd.to_datetime(
        df["startTime"], utc=True
    )

    df = df.pivot(
        index="startTime",
        columns="fuelType",
        values="generation"
    ).reset_index()

    df.columns.name = None

    return df.sort_values(
        "startTime"
    ).reset_index(drop=True)


In [60]:
generation_frames = []
failed_dates = []

for date in dates:
    try:
        daily_df = generation_day(date)
        generation_frames.append(daily_df)
    except requests.exceptions.RequestException:
        failed_dates.append(date)

generation_history = pd.concat(
    generation_frames,
    ignore_index=True
).sort_values("startTime").reset_index(drop=True)

print("Collected observations:", len(generation_history))
print("Failed dates:", len(failed_dates))


Collected observations: 35088
Failed dates: 0


### 5.4 Validate, clean and save

In [61]:
missing_generation_times = expected_times.difference(
    generation_history["startTime"]
)

print("Expected rows:", len(expected_times))
print("Actual rows:", len(generation_history))
print("Missing timestamps:", len(missing_generation_times))
print("Duplicate timestamps:", generation_history["startTime"].duplicated().sum())

print("\nMissing values:")
print(generation_history.isna().sum())

print("\nChronological:", generation_history["startTime"].is_monotonic_increasing)

print("\nGeneration summary:")
print(generation_history.drop(columns="startTime").describe())

# %%
missing_generation_times

# %%
full_timeline = pd.DataFrame({
    "startTime": expected_times
})

generation_clean = full_timeline.merge(
    generation_history,
    on="startTime",
    how="left",
    validate="one_to_one"
)

# %%
print("Shape:", generation_clean.shape)

print("\nMissing values:")
print(generation_clean.isna().sum())

print("\nDuplicate timestamps:",
      generation_clean["startTime"].duplicated().sum())

print("\nChronological:",
      generation_clean["startTime"].is_monotonic_increasing)

# %%
processed_path = (
    PROCESSED_DIR / "electricity_generation_2024_2025.csv"
)

generation_clean.to_csv(
    processed_path,
    index=False
)


Expected rows: 35088
Actual rows: 35088
Missing timestamps: 0
Duplicate timestamps: 0

Missing values:
startTime        0
BIOMASS          0
CCGT             0
COAL             0
INTELEC          0
INTEW            0
INTFR            0
INTIFA2          0
INTIRL           0
INTNED           0
INTNEM           0
INTNSL           0
INTVKL           0
NPSHYD           0
NUCLEAR          0
OCGT             0
OIL              0
OTHER            0
PS               0
WIND             0
INTGRNL      10466
dtype: int64

Chronological: True

Generation summary:
            BIOMASS          CCGT          COAL       INTELEC         INTEW  \
count  35088.000000  35088.000000  35088.000000  35088.000000  35088.000000   
mean    2207.106789   8538.517841     89.443570    631.480050   -249.090230   
std      801.078164   5610.689543    240.937253    461.992122    253.960952   
min        0.000000      0.000000      0.000000  -1024.000000   -588.000000   
25%     1555.000000   3887.750000      0.000000 